# NBME 資料前處理

1. 合併 `train.csv`、`patient_notes.csv`、`features.csv`
2. 將 `location` 轉成 character spans
3. 用 spans 從 `pn_history` 擷取文字並比對 `annotation`
4. 建立統一格式並存成 `.pkl`(包含 5-fold 訓練切分)

In [2]:
from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except ImportError:
    from sklearn.model_selection import GroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = False

In [3]:
DATA_DIR = Path("data/nbme-score-clinical-patient-notes")
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
PATIENT_NOTES_PATH = DATA_DIR / "patient_notes.csv"
FEATURES_PATH = DATA_DIR / "features.csv"
OUTPUT_PATH = OUTPUT_DIR / "train_preprocessed_5fold.pkl"
TEST_OUTPUT_PATH = OUTPUT_DIR / "test_preprocessed.pkl"

N_SPLITS = 5
RANDOM_STATE = 42

## Step 1: 讀取並合併三個資料表

In [5]:
train = pd.read_csv(
    TRAIN_PATH,
    dtype={"id": "string", "pn_num": "string", "feature_num": "string"},
)
patient_notes = pd.read_csv(
    PATIENT_NOTES_PATH,
    dtype={"pn_num": "string"}, # 有前導 0 -> 固定成字串
)
features = pd.read_csv(
    FEATURES_PATH,
    dtype={"feature_num": "string"},    # 有前導 0 -> 固定成字串
)

df = (
    train
    .merge(patient_notes, on=["case_num", "pn_num"], how="left", validate="many_to_one")
    .merge(features, on=["case_num", "feature_num"], how="left", validate="many_to_one")
)

print(train.shape, patient_notes.shape, features.shape, df.shape)
display(df.head())

(14300, 6) (42146, 3) (143, 3) (14300, 8)


,id,case_num,pn_num,feature_num,annotation,location,pn_history,feature_text
0,00016_000,0,00016,000,['dad with recent heart attcak'],['696 724'],HPI: 17yo M presents with palpitations. Patien...,Family-history-of-MI-OR-Family-history-of-myoc...
1,00016_001,0,00016,001,"['mom with ""thyroid disease']",['668 693'],HPI: 17yo M presents with palpitations. Patien...,Family-history-of-thyroid-disorder
2,00016_002,0,00016,002,['chest pressure'],['203 217'],HPI: 17yo M presents with palpitations. Patien...,Chest-pressure
3,00016_003,0,00016,003,"['intermittent episodes', 'episode']","['70 91', '176 183']",HPI: 17yo M presents with palpitations. Patien...,Intermittent-symptoms
4,00016_004,0,00016,004,['felt as if he were going to pass out'],['222 258'],HPI: 17yo M presents with palpitations. Patien...,Lightheaded


In [4]:
missing_summary = df[["pn_history", "feature_text"]].isna().sum()
print(missing_summary)

if missing_summary.any():
    raise ValueError("Merge 後有缺失的 pn_history 或 feature_text，請檢查 key 欄位。")

pn_history      0
feature_text    0
dtype: int64


## Step 2: 將 location 轉成 char_spans

- `span_groups`：step3 驗證 annotation 用，兩層`list[list[tuple]]`，會保留 annotation 對齊關係
- `char_spans`：訓練資料使用，一層 `list[tuple[int, int]]`，允許同一筆資料有多段不連續 span

In [5]:
def parse_literal_list(value):
    """把字串格式的 list 轉成 list"""
    if pd.isna(value) or value == "":
        return []
    return ast.literal_eval(value)


def parse_span_group(location_item):
    """把location欄位： '10 15;20 25' 轉成tuple: [(10, 15), (20, 25)]"""
    spans = []

    for span in location_item.split(";"):
        start, end = span.split()
        start, end = int(start), int(end)

        if start > end:
            raise ValueError(f"span 的 start 比 end 大：{location_item}")

        spans.append((start, end))

    return spans


def parse_span_groups(location):
    """保留每個 annotation 對應到哪些 span 給 step3 檢查用"""
    return [parse_span_group(item) for item in parse_literal_list(location)]


def flatten_span_groups(span_groups):
    """把兩層 span 攤平成一層，當作最後的 char_spans"""
    return [span for span_group in span_groups for span in span_group]


assert parse_span_group("10 15;20 25") == [(10, 15), (20, 25)]
assert flatten_span_groups([[(10, 15), (20, 25)], [(30, 35)]]) == [(10, 15), (20, 25), (30, 35)]

In [6]:
df["annotation_text"] = df["annotation"].apply(parse_literal_list)
df["span_groups"] = df["location"].apply(parse_span_groups)
df["char_spans"] = df["span_groups"].apply(flatten_span_groups)

length_mismatch = df[df["annotation_text"].str.len() != df["span_groups"].str.len()]
print(f"annotation 和 location 數量對不起來的筆數：{len(length_mismatch)}")

if len(length_mismatch) > 0:
    display(length_mismatch[["id", "annotation", "location", "annotation_text", "span_groups"]].head(20))
    raise ValueError("annotation 和 location 數量對不起來，請檢查原始資料。")

display(df[["id", "annotation_text", "span_groups", "char_spans"]].head())

annotation 和 location 數量對不起來的筆數：0


,id,annotation_text,span_groups,char_spans
0,00016_000,[dad with recent heart attcak],"[[(696, 724)]]","[(696, 724)]"
1,00016_001,"[mom with ""thyroid disease]","[[(668, 693)]]","[(668, 693)]"
2,00016_002,[chest pressure],"[[(203, 217)]]","[(203, 217)]"
3,00016_003,"[intermittent episodes, episode]","[[(70, 91)], [(176, 183)]]","[(70, 91), (176, 183)]"
4,00016_004,[felt as if he were going to pass out],"[[(222, 258)]]","[(222, 258)]"


## Step 3: 用 span_groups 回到 pn_history 擷取文字並確認 annotation

`span_groups` 保留每個 annotation 對應哪些 span，所以可正確檢查不連續 span。比對時會做基本 normalization：轉小寫、壓縮空白。

In [9]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_from_spans(text, span_group):
    return " ".join(text[start:end] for start, end in span_group)


def extract_all_annotations(row):
    return [extract_from_spans(row["pn_history"], span_group) for span_group in row["span_groups"]]


def is_annotation_match(row):
    extracted = row["extracted_annotation_text"]
    expected = row["annotation_text"]
    if len(extracted) != len(expected):
        return False
    return all(normalize_text(a) == normalize_text(b) for a, b in zip(extracted, expected))


df["extracted_annotation_text"] = df.apply(extract_all_annotations, axis=1)
df["annotation_match"] = df.apply(is_annotation_match, axis=1)

compare_result = df[[
    "id",
    "annotation_text",
    "extracted_annotation_text",
    "span_groups",
    "char_spans",
    "annotation_match",
]].copy()

match_rate = df["annotation_match"].mean()
mismatch_count = (~df["annotation_match"]).sum()

print(f"比對正確率：{match_rate:.4%}")
print(f"比對不一致筆數：{mismatch_count}")

print("前 20 筆比對結果：")
display(compare_result.head(20))

print("比對不一致的資料：")
display(compare_result[~compare_result["annotation_match"]].head(20))

比對正確率：100.0000%
比對不一致筆數：0
前 20 筆比對結果：


,id,annotation_text,extracted_annotation_text,span_groups,char_spans,annotation_match
0,00016_000,[dad with recent heart attcak],[dad with recent heart attcak],"[[(696, 724)]]","[(696, 724)]",True
1,00016_001,"[mom with ""thyroid disease]","[mom with ""thyroid disease]","[[(668, 693)]]","[(668, 693)]",True
2,00016_002,[chest pressure],[chest pressure],"[[(203, 217)]]","[(203, 217)]",True
3,00016_003,"[intermittent episodes, episode]","[intermittent episodes, episode]","[[(70, 91)], [(176, 183)]]","[(70, 91), (176, 183)]",True
4,00016_004,[felt as if he were going to pass out],[felt as if he were going to pass out],"[[(222, 258)]]","[(222, 258)]",True
5,00016_005,[],[],[],[],True
6,00016_006,"[adderall, adderrall, adderrall]","[adderall, adderrall, adderrall]","[[(321, 329)], [(404, 413)], [(652, 661)]]","[(321, 329), (404, 413), (652, 661)]",True
7,00016_007,[],[],[],[],True
8,00016_008,[],[],[],[],True
9,00016_009,"[palpitations, heart beating/pounding]","[palpitations, heart beating/pounding]","[[(26, 38)], [(96, 118)]]","[(26, 38), (96, 118)]",True


比對不一致的資料：


,id,annotation_text,extracted_annotation_text,span_groups,char_spans,annotation_match


## Step 4: 整理訓練集統一資料格式並建立 5-fold

Fold 切分以 `pn_num` 作為 group(避免同一份 patient note 出現在不同 fold)

In [10]:
df["fold"] = -1

if HAS_STRATIFIED_GROUP_KFOLD:
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(df, y=df["case_num"], groups=df["pn_num"])
else:
    print("StratifiedGroupKFold not available; falling back to GroupKFold.")
    splitter = GroupKFold(n_splits=N_SPLITS)
    split_iter = splitter.split(df, groups=df["pn_num"])

for fold, (_, valid_idx) in enumerate(split_iter):
    df.loc[valid_idx, "fold"] = fold

if (df["fold"] < 0).any():
    raise ValueError("有資料列沒有被分配到 fold，請檢查切分流程。")

fold_summary = pd.crosstab(df["fold"], df["case_num"])
display(fold_summary)
print(df["fold"].value_counts().sort_index())

case_num,0,1,2,3,4,5,6,7,8,9
fold,,,,,,,,,,
0,299,247,323,352,210,306,264,171,378,289
1,260,273,357,304,190,396,216,189,360,323
2,273,247,340,304,190,378,240,189,342,357
3,247,247,374,336,170,396,240,189,324,357
4,221,286,306,304,240,324,240,162,396,374


fold
0    2839
1    2868
2    2860
3    2880
4    2853
Name: count, dtype: int64


In [ ]:
output_columns = [
    "id",
    "case_num",
    "pn_num",
    "feature_num",
    "pn_history",
    "feature_text",
    "annotation_text",
    "char_spans",
    "fold",
]

processed = df[output_columns].copy()

processed.to_pickle(OUTPUT_PATH)    # Python 專用的資料儲存格式

print(f"Saved: {OUTPUT_PATH}")
print(processed.shape)
display(processed.head())

Saved: data/nbme-score-clinical-patient-notes/processed/train_preprocessed_5fold.pkl
(14300, 9)


,id,case_num,pn_num,feature_num,pn_history,feature_text,annotation_text,char_spans,fold
0,00016_000,0,00016,000,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-MI-OR-Family-history-of-myoc...,[dad with recent heart attcak],"[(696, 724)]",0
1,00016_001,0,00016,001,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-thyroid-disorder,"[mom with ""thyroid disease]","[(668, 693)]",0
2,00016_002,0,00016,002,HPI: 17yo M presents with palpitations. Patien...,Chest-pressure,[chest pressure],"[(203, 217)]",0
3,00016_003,0,00016,003,HPI: 17yo M presents with palpitations. Patien...,Intermittent-symptoms,"[intermittent episodes, episode]","[(70, 91), (176, 183)]",0
4,00016_004,0,00016,004,HPI: 17yo M presents with palpitations. Patien...,Lightheaded,[felt as if he were going to pass out],"[(222, 258)]",0


In [12]:
# Quick reload check
reloaded = pd.read_pickle(OUTPUT_PATH)
print(reloaded.shape)
display(reloaded.head())

(14300, 9)


,id,case_num,pn_num,feature_num,pn_history,feature_text,annotation_text,char_spans,fold
0,00016_000,0,00016,000,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-MI-OR-Family-history-of-myoc...,[dad with recent heart attcak],"[(696, 724)]",0
1,00016_001,0,00016,001,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-thyroid-disorder,"[mom with ""thyroid disease]","[(668, 693)]",0
2,00016_002,0,00016,002,HPI: 17yo M presents with palpitations. Patien...,Chest-pressure,[chest pressure],"[(203, 217)]",0
3,00016_003,0,00016,003,HPI: 17yo M presents with palpitations. Patien...,Intermittent-symptoms,"[intermittent episodes, episode]","[(70, 91), (176, 183)]",0
4,00016_004,0,00016,004,HPI: 17yo M presents with palpitations. Patien...,Lightheaded,[felt as if he were going to pass out],"[(222, 258)]",0


## Step 5: 整理 test inference 資料

`test.csv` 沒有 `annotation` / `location`，所以不會產生 `char_spans`、`annotation_text`、`fold`。


In [9]:
test = pd.read_csv(
    TEST_PATH,
    dtype={"id": "string", "pn_num": "string", "feature_num": "string"},
)

test_df = (
    test
    .merge(patient_notes, on=["case_num", "pn_num"], how="left", validate="many_to_one")
    .merge(features, on=["case_num", "feature_num"], how="left", validate="many_to_one")
)

test_missing_summary = test_df[["pn_history", "feature_text"]].isna().sum()
print("test_missing_summary :\n",test_missing_summary)

if test_missing_summary.any():
    raise ValueError("Test merge 後有缺失的 pn_history 或 feature_text，請檢查 key 欄位。")

test_output_columns = [
    "id",
    "case_num",
    "pn_num",
    "feature_num",
    "pn_history",
    "feature_text",
]

test_processed = test_df[test_output_columns].copy()
test_processed.to_pickle(TEST_OUTPUT_PATH)

print(f"Saved: {TEST_OUTPUT_PATH}")
print(test_processed.shape)
display(test_processed.head())


test_missing_summary :
 pn_history      0
feature_text    0
dtype: int64
Saved: data/nbme-score-clinical-patient-notes/processed/test_preprocessed.pkl
(5, 6)


,id,case_num,pn_num,feature_num,pn_history,feature_text
0,00016_000,0,00016,000,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-MI-OR-Family-history-of-myoc...
1,00016_001,0,00016,001,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-thyroid-disorder
2,00016_002,0,00016,002,HPI: 17yo M presents with palpitations. Patien...,Chest-pressure
3,00016_003,0,00016,003,HPI: 17yo M presents with palpitations. Patien...,Intermittent-symptoms
4,00016_004,0,00016,004,HPI: 17yo M presents with palpitations. Patien...,Lightheaded


In [10]:
# Quick reload check for test preprocess
test_reloaded = pd.read_pickle(TEST_OUTPUT_PATH)
print(test_reloaded.shape)
display(test_reloaded.head())


(5, 6)


,id,case_num,pn_num,feature_num,pn_history,feature_text
0,00016_000,0,00016,000,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-MI-OR-Family-history-of-myoc...
1,00016_001,0,00016,001,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-thyroid-disorder
2,00016_002,0,00016,002,HPI: 17yo M presents with palpitations. Patien...,Chest-pressure
3,00016_003,0,00016,003,HPI: 17yo M presents with palpitations. Patien...,Intermittent-symptoms
4,00016_004,0,00016,004,HPI: 17yo M presents with palpitations. Patien...,Lightheaded
